In [1]:
# Football Match Outcome Prediction
## Goal: predict whether a football match ends in a home win, away win, or draw, using historical international match data (1872–2017).
## This is a baseline model built as a first end-to-end ML project — data loading, cleaning, feature encoding, training, and evaluation.

In [2]:
## 1. Load and look at the data
## The first step is to load the dataset and check what I'm working with.

import pandas as pd
df = pd.read_csv('results.csv')
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


In [3]:
## 2. Basic data checks
## Check the dataset size and whether any data is missing before doing anything else.

df.shape
df.isnull().sum()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 49547 entries, 0 to 49546
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        49547 non-null  str  
 1   home_team   49547 non-null  str  
 2   away_team   49547 non-null  str  
 3   home_score  49547 non-null  int64
 4   away_score  49547 non-null  int64
 5   tournament  49547 non-null  str  
 6   city        49547 non-null  str  
 7   country     49547 non-null  str  
 8   neutral     49547 non-null  bool 
dtypes: bool(1), int64(2), str(6)
memory usage: 3.1 MB


In [4]:
## No missing values in any column, and about 49,500 rows. One thing to fix later: `date` is stored as a string, not an actual date type 
## not a problem for this baseline, but would matter if I add time-based features later.

In [5]:
## 3. Create the target variable
## The dataset has home_score and away_score, but no direct "who won" column, so I built one
## comparing the two scores to label each match as a home win, away win, or draw.

def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'
df['result'] = df.apply(get_result, axis=1)
df['result'].value_counts()

result
home_win    24276
away_win    14010
draw        11261
Name: count, dtype: int64

In [7]:
from sklearn.preprocessing import LabelEncoder

le_home = LabelEncoder()
le_away = LabelEncoder()

df['home_team_enc'] = le_home.fit_transform(df['home_team'])
df['away_team_enc'] = le_away.fit_transform(df['away_team'])
df['neutral_enc'] = df['neutral'].astype(int)

df[['home_team', 'home_team_enc', 'away_team', 'away_team_enc', 'neutral', 'neutral_enc']].head()

,home_team,home_team_enc,away_team,away_team_enc,neutral,neutral_enc
0,Scotland,252,England,89,False,0
1,England,87,Scotland,247,False,0
2,Scotland,252,England,89,False,0
3,England,87,Scotland,247,False,0
4,Scotland,252,England,89,False,0


In [8]:
## Home teams win close to half of all matches, away teams win about 28%, and draws are around 23%.
## This confirms there's a real home advantage in the data, which is useful to know before building the model.

In [9]:
## 4. Attempt 1 Label encoding team names
## First approach: convert team names into numbers using LabelEncoder so the model can use them.
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Features and target
X = df[['home_team_enc', 'away_team_enc', 'neutral_enc']]
y = df['result']

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit baseline model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Predict and evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.48980827447023206
              precision    recall  f1-score   support

    away_win       0.00      0.00      0.00      2790
        draw       0.00      0.00      0.00      2266
    home_win       0.49      1.00      0.66      4854

    accuracy                           0.49      9910
   macro avg       0.16      0.33      0.22      9910
weighted avg       0.24      0.49      0.32      9910



C:\Users\ASH\DataScience\football-model\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASH\DataScience\football-model\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\ASH\DataScience\football-model\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [10]:
## Accuracy came out to 49%, but something was clearly off: precision and recall were 0 for away_win and draw, and I got an "UndefinedMetricWarning" in the output.
## Reading through the warning message, it pointed to the model never predicting those classes at all 
## it was just guessing home_win every single time, which happened to match the base rate closely enough to look like it was working.
## The actual problem was in how I encoded the teams. LabelEncoder assigns arbitrary numbers (e.g. Scotland = 252, England = 89), and the model was 
## treating that like a real numeric scale, as if higher numbers meant something, which they don't. Team identity isn't ordered data.

In [16]:
## 5. Attempt 2 One-hot encoding instead
## Switched to one-hot encoding, which gives each team its own true/false column instead of a single arbitrary number, so there's no false ranking between teams.

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Step 1: turn team names into yes/no columns instead of numbers
X = pd.get_dummies(df[['home_team', 'away_team', 'neutral_enc']], columns=['home_team', 'away_team'])
y = df['result']

# Step 2: split into training data and testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Step 4: check how well it did
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy: 0.5779011099899092
              precision    recall  f1-score   support

    away_win       0.53      0.55      0.54      2790
        draw       0.31      0.04      0.07      2266
    home_win       0.61      0.85      0.71      4854

    accuracy                           0.58      9910
   macro avg       0.48      0.48      0.44      9910
weighted avg       0.52      0.58      0.51      9910



In [17]:
## 6. Results
## Accuracy improved to about 58%, and this time the model is actually distinguishing between outcomes instead of just guessing one class 85% recall on home wins, 55% on away wins.
## Draws are still the weak point only 4% recall. Trial and error is basically the whole process here: draws are genuinely harder to predict because they don't follow team-strength patterns as cleanly as wins do
## two evenly matched teams are close to a coin flip regardless of who's stronger on paper.